In [1]:
# enable autoreload so imported modules refresh automatically after installs/edits
%load_ext autoreload
%autoreload 2

In [5]:
from pathlib import Path
import re
from collections import Counter
import pandas as pd

folder = Path(r"C:\Users\davib\Desktop\MSc_DataScience\thesis\gpt_struct_me\resources\lusa_news")
files = sorted(folder.glob("*.txt"))

word_counts = Counter()

for file in files:
    text = file.read_text(encoding="utf-8", errors="ignore")
    tokens = re.findall(r"\b\w+\b", text.lower())
    word_counts.update(tokens)

word_freq_df_pt = pd.DataFrame(
    word_counts.items(),
    columns=["word", "frequency"]
).sort_values("frequency", ascending=False).reset_index(drop=True)
word_freq_df_pt
word_freq_df_pt.to_csv(Path.cwd() / "word_freq_lusa_pt.csv", index=False)

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter
import pandas as pd

en_folder = Path(r"C:\Users\davib\Desktop\MSc_DataScience\thesis\gpt_struct_me\resources\lusa_en")
json_files = sorted(en_folder.glob("*.json"))

def extract_sofa_strings(obj):
    if isinstance(obj, dict):
        if "sofaString" in obj and isinstance(obj["sofaString"], str):
            yield obj["sofaString"]
        for value in obj.values():
            yield from extract_sofa_strings(value)
    elif isinstance(obj, list):
        for item in obj:
            yield from extract_sofa_strings(item)

sofa_texts = []

for file in json_files:
    try:
        data = json.loads(file.read_text(encoding="utf-8", errors="ignore"))
        sofa_texts.extend(extract_sofa_strings(data))
    except Exception as e:
        print(f"Could not parse {file.name}: {e}")

text = "\n".join(sofa_texts)

tokens = re.findall(r"\b\w+\b", text.lower())
word_counts = Counter(tokens)

word_freq_df_en = pd.DataFrame(
    word_counts.items(),
    columns=["word", "frequency"]
).sort_values("frequency", ascending=False).reset_index(drop=True)
word_freq_df_en
word_freq_df_en.to_csv(Path.cwd() / "word_freq_lusa_en.csv", index=False)